# Day 13 · 小项目:零售订单 EDA(W2 第 6 天)

目标:拿一份 500 行的零售订单数据(`../datasets/retail-orders.csv`),从 0 到结论完整做一遍。

今日节奏:40min 学习 + 15min 动手 + 5min 自检。

> 打开方式:JupyterLab 文件树里进 `ai-learning/练习/` 双击本文件,逐格 Shift+Enter。


In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "| pandas", pd.__version__)
print("环境 OK!开始今天的练习 →")

orders = pd.read_csv(os.path.join("..", "datasets", "retail-orders.csv"), parse_dates=["date"])
print("形状:", orders.shape)
print(orders.head())
print("缺失:\n", orders.isna().sum())


## 任务 1:清洗与概览

样本里埋了点小问题:category 有缺失、文本有空格。按 Day 10 的套路清掉,再看数值范围。


In [ ]:
orders = orders.copy()
orders["category"] = orders["category"].fillna("Other").str.strip()
orders["city"] = orders["city"].fillna("Unknown").str.strip()
print("清洗后缺失:", orders.isna().sum().sum())
print("日期范围:", orders["date"].min().date(), "→", orders["date"].max().date())
print(orders[["amount"]].describe().round(1))


## 任务 2:时间维度(每天卖多少?)

按天聚合销售额,画折线看趋势;再把 7 日移动平均叠上去。


In [ ]:
daily = orders.groupby("date")["amount"].sum()
daily.plot(figsize=(7, 3), alpha=0.5, label="daily", grid=True)
daily.rolling(7).mean().plot(label="7-day avg", linewidth=2)
plt.title("Daily revenue")
plt.ylabel("Amount")
plt.legend()
plt.show()


## 任务 3:类别维度(什么最赚钱?)

每个类别的销售额和订单数,再用柱状图 + 饼图看占比。


In [ ]:
cat = orders.groupby("category").agg(销售额=("amount", "sum"), 订单数=("order_id", "count"))
print(cat.sort_values("销售额", ascending=False).round(0))

plt.figure(figsize=(8, 3.5))
plt.subplot(1, 2, 1)
cat["销售额"].plot(kind="bar", color="teal")
plt.xticks(rotation=30)
plt.title("Revenue by category")

plt.subplot(1, 2, 2)
cat["销售额"].plot(kind="pie", autopct="%1.1f%%")
plt.ylabel("")
plt.title("Category share")

plt.tight_layout()
plt.show()


## 任务 4:城市维度(谁是前 10?)

横向柱状图 barh 最适合城市名这类长标签。


In [ ]:
city = orders.groupby("city")["amount"].sum().sort_values().tail(10)
city.plot(kind="barh", color="purple")
plt.xlabel("Revenue")
plt.title("Top 10 cities")
plt.show()


## 任务 5:客单价与复购

- 每笔订单金额的分布(直方图)
- 每个客户下了几单?复购客户占多少?前 5 大客户是谁?


In [ ]:
orders["amount"].hist(bins=30, color="green", alpha=0.7)
plt.xlabel("Order amount")
plt.ylabel("Count")
plt.title("Order size distribution")
plt.show()

per_customer = orders.groupby("customer_id")["order_id"].count()
print("复购客户占比:", round((per_customer > 1).mean() * 100, 1), "%")
print("前 5 大客户:\n", per_customer.sort_values(ascending=False).head())


## 任务 6:写三句话结论 🔥

用刚才的数字各写一句,模板:
1. 时间上……(趋势怎么样)
2. 结构上……(哪个类别/城市贡献最大)
3. 建议……(把预算/补货投到哪)


In [ ]:
top_cat = cat["销售额"].idxmax()
top_city = city.idxmax()
best_day = daily.idxmax()
print("1. 时间:销售额最高的一天是", best_day.date(), ",整体有周期性波动")
print("2. 结构:最赚钱的类别是", top_cat, ",最高城市是", top_city)
print("3. 建议:围绕", top_cat, "和", top_city, "做补货与促销,验证能否进一步提升")


## 自检清单(5 问,答不上就回看今天的格子)

1. EDA 项目从哪开始? → 读数据 + shape/head/缺失体检
2. 趋势图怎么画? → 按天聚合后 plot,再叠 rolling(7).mean()
3. 类别占比用什么图? → pie(五六个类别以内)
4. 城市前 10 用什么图? → barh 横向柱状图
5. 结论怎么写? → 时间 / 结构 / 建议,三句话,每句都有数字支撑

## 📝 收盘动作

```powershell
cd D:\01_Study\ai-learning
git add -A; git commit -m "day13: retail eda"; git push
```

然后跟助手说"生成日志"。
